In [2]:
import os 
import json 
from dotenv import load_dotenv
load_dotenv()
from openai import OpenAI
open_ai_api_key = os.getenv("OPENAI_API_KEY")
from connection_db import connection_with_oracle

In [3]:
connection = connection_with_oracle()
cursor = connection.cursor()

Attempting secure MTS Connection
('Connected!',)


In [8]:
# Fetching the data
import urllib.request
import time 

url = "https://raw.githubusercontent.com/Denis2054/RAG-Driven-Generative-AI-2nd-Edition/main/enterprise_data/talent_acquisition/hr_data.json"
output_name = 'hr_data.json'

urllib.request.urlretrieve(url , output_name)
print("Download complete")

# Let the file system settle down
print("Let file system settle down")
time.sleep(5)

print("\n🎉 SUCCESS!")

Download complete
Let file system settle down

🎉 SUCCESS!


In [10]:
# Load HR Data in memory
hr_data = {}
file_path = "hr_data.json"

if os.path.exists(file_path):
    try:
        with open(file_path , "r" , encoding="utf-8") as file:
            hr_data = json.load(file)
        print(f"✅ HR Data Loaded Successfully")
        print(f"   - Candidates found: {len(hr_data.get('candidates', []))}")
        print(f"   - Recruitment Rules found: {len(hr_data.get('rules', []))}")
    
    except json.JSONDecodeError as e:
        print("Error in data loading from file to memory")
else:
    print("No File Exist")


✅ HR Data Loaded Successfully
   - Candidates found: 5
   - Recruitment Rules found: 3


In [11]:
# Chunking process
import tiktoken
tokenizre = tiktoken.get_encoding('cl100k_base')

def chunk_text(text , chunk_size = 400 , overlap = 50):
    """ 
        Chunk Token based on the token count and overlap 
    """
    tokens = tokenizre.encode(text)
    chunks = []
    for i in range(0 , len(tokens), chunk_size - overlap):
        chunk_tokens = tokens[i : i + chunk_size]
        chunk_text = tokenizre.decode(chunk_tokens)
        chunk_text = chunk_text.replace('\n' , " ")
        if chunk_text:
            chunks.append(chunk_text)
    return chunks

In [13]:
# Embeddings 
from tenacity import retry, wait_random_exponential, stop_after_attempt
client = OpenAI(api_key=open_ai_api_key)
EMBEDDING_MODEL = "text-embedding-3-small"
EMBEDDING_DIM = 1536 # Dimension for text-embedding-3-small
GENERATION_MODEL = "gpt-5.2"

@retry(wait=wait_random_exponential(min=1, max=60), stop=stop_after_attempt(6))
def get_embeddings_batch(texts, model=EMBEDDING_MODEL):
    """
    Generates embeddings for a batch of texts using OpenAI.
    """

    texts = [t.replace("\n", " ") for t in texts]

    response = client.embeddings.create(input=texts, model=model)

    return [item.embedding for item in response.data]

In [14]:
# Inserting the data

# Lets start with candidate

candidates = hr_data.get("candidates" , [])
for cand in candidates:
    cursor.execute("""
            MERGE INTO candidate_pool target
            USING (SELECT :candidate_id AS id FROM dual) source
            ON (target.candidate_id = source.id)
            WHEN NOT MATCHED THEN
                INSERT (candidate_id, full_name, years_experience, salary_expectation, skills, summary)
                VALUES (:candidate_id, :full_name, :years_experience, :salary_expectation, :skills, :summary)
        """, {
            "candidate_id": cand['candidate_id'],
            "full_name": cand['full_name'],
            "years_experience": cand['years_experience'],
            "salary_expectation": cand['salary_expectation'],
            "skills": cand['skills'],
            "summary": cand['summary']
        })
print(f"{len(candidates)} added to the candidate_pool")


# Now with Rules

rules = hr_data.get("rules" , [])
for rule in rules:
        cursor.execute("""
            MERGE INTO recruitment_rules target
            USING (SELECT :rule_id AS id FROM dual) source
            ON (target.rule_id = source.id)
            WHEN NOT MATCHED THEN
                INSERT (rule_id, agent_persona, evaluation_criteria)
                VALUES (:rule_id, :agent_persona, :evaluation_criteria)
        """, {
            "rule_id": rule['rule_id'],
            "agent_persona": rule['agent_persona'],
            "evaluation_criteria": rule['evaluation_criteria']
        })

print(f"{len(rules)} added to the recruitment_rules table")

connection.commit()
print("\n✅ SQL Data Added.")


5 added to the candidate_pool
3 added to the recruitment_rules table

✅ SQL Data Added.


In [ ]:
# Vectorization of Tables 
# Fetch the raw make embeddings and update the raw with vector 
import oracledb

cursor = connection.cursor()
# Start with candidate_pool
cursor.execute("SELECT candidate_id, summary FROM candidate_pool WHERE resume_vector IS NULL")
rows_to_process = cursor.fetchall()

if rows_to_process:
    summaries = [row[1].read() for row in rows_to_process]
    vectors = get_embeddings_batch(summaries)

    for i, (cand_id, _) in enumerate(rows_to_process):
            cursor.setinputsizes(vec=oracledb.DB_TYPE_VECTOR)
            cursor.execute("""
                UPDATE candidate_pool
                SET resume_vector = :vec
                WHERE candidate_id = :id
            """, {"vec": vectors[i], "id": cand_id})
    
    connection.commit()
    print("--- Candidates Update with Vector")
else:
      print("--- No Candidate need vectorization")


# Recruitment Rules
cursor.execute("SELECT rule_id, agent_persona || ' ' || evaluation_criteria FROM recruitment_rules WHERE rule_vector IS NULL")
rows_to_process = cursor.fetchall()

if rows_to_process:
    # Check the text is string of lob(CLOB)
    text = [] 
    for row in rows_to_process:
        data = row[1]
        if hasattr(data , 'read'):
            text.append(data.read()) # For CLOB
        else:
            text.append(data) # For string 

    vectors = get_embeddings_batch(text) 

    for i, (rule_id, _) in enumerate(rows_to_process):
            cursor.setinputsizes(vec=oracledb.DB_TYPE_VECTOR)
            cursor.execute("""
                UPDATE recruitment_rules
                SET rule_vector = :vec
                WHERE rule_id = :id
            """, {"vec": vectors[i], "id": rule_id})

    connection.commit()
    print("--- Rules Update with Vector")
else:
    print("--- No Row required any Vectorization")   

print("\n Vectorization Done 💯")      
                  

--- Candidates Update  with Vector
Rules Vector Added

 Vectorization Done 💯


In [23]:
# final Hybrid Query (SQL + Vector)

# Define hybrid query
# We want a leader with a proper budget
user_query = "Leadership and teambuilding Experience"
max_budget = 170000

print(f"🔎 Query: '{user_query}'")
print(f"💰 Constraint: Salary <= ${max_budget}\n")

# First generate embeddings for the user query
query_vector = get_embeddings_batch([user_query])[0]

# Execute Hybrid SQL
# Filter by SQL Salary and order by vector distance

cursor.setinputsizes(v=oracledb.DB_TYPE_VECTOR)
cursor.execute("""
    SELECT candidate_id, full_name, salary_expectation, summary,
           VECTOR_DISTANCE(resume_vector, :v, DOT) as similarity
    FROM candidate_pool
    WHERE salary_expectation <= :budget
    ORDER BY similarity DESC
    FETCH FIRST 3 ROWS ONLY
""", {"v": query_vector, "budget": max_budget})

results = cursor.fetchall()

print("Hybrid Search Result")
for r in results:
    cand_id, name, salary, summary_lob, score = r

    # FIX: Convert LOB to string
    summary_text = summary_lob.read()

    print(f"Candidate: {name} (ID: {cand_id})")
    print(f"Salary: ${salary:,}")
    print(f"Match Score: {score:.4f}")
    print(f"Summary Snippet: {summary_text[:100]}...")
    print("-" * 50)

🔎 Query: 'Leadership and teambuilding Experience'
💰 Constraint: Salary <= $170000

Hybrid Search Result
Candidate: Riley S. (ID: CAND_004)
Salary: $140,000
Match Score: -0.2069
Summary Snippet: Backend Java Developer focused on banking transaction systems. Solid experience with Oracle Database...
--------------------------------------------------
Candidate: Jordan L. (ID: CAND_002)
Salary: $85,000
Match Score: -0.2180
Summary Snippet: Junior Data Scientist recently graduated with a Masters in AI. Strong academic background in machine...
--------------------------------------------------
Candidate: Alex V. (ID: CAND_001)
Salary: $165,000
Match Score: -0.2896
Summary Snippet: Senior Full Stack Engineer with deep expertise in cloud-native architectures. specialized in buildin...
--------------------------------------------------
